# MIRAGE++ — Notebook 1: Theory and Motivation

## Why Classical Linear Regression Fails in Finance

Linear regression underlies virtually every quantitative finance model: factor attribution, signal combination, portfolio construction, volatility forecasting. Yet OLS has three **structural failure modes** that are not statistical noise but geometric properties.

### Failure 1: Unconstrained Weights Are Economically Meaningless

OLS minimises $\frac{1}{m}\|X\theta - y\|^2$ over all $\theta \in \mathbb{R}^n$. The solution routinely produces negative weights (short positions), weights greater than 1 (leverage), and weights summing to arbitrary values (no budget constraint). Post-hoc projection (clip, renormalise) is **not a fix** — the projected vector is no longer the solution to any well-defined problem, destroying all theoretical guarantees.

### Failure 2: Concentration Under Correlated Features

When features are correlated, OLS places extreme weight on one representative from each correlated cluster and near-zero on the rest. This is a mathematical consequence of the OLS objective, not sampling noise. Lasso makes it worse by explicitly inducing sparsity. A portfolio of 11 sector ETFs is not diversified if 85% of the weight sits on XLK.

### Failure 3: Post-hoc Projection Breaks All Guarantees

Regret bounds for projected gradient descent require projection at *every* iteration. Training unconstrained and projecting at test time produces no algorithm with a name — and no convergence certificate.

---

## The MIRAGE++ Formulation

$$\min_{\theta \in \Delta_{n-1}} \mathcal{L}(\theta) = \underbrace{\frac{1}{m}\|X\theta - y\|^2}_{\text{prediction error}} - \underbrace{\lambda H(\theta)}_{\text{diversity bonus}}$$

where $\Delta_{n-1} = \{\theta \geq 0 : \sum_i \theta_i = 1\}$ is the probability simplex, $H(\theta) = -\sum_i \theta_i \log \theta_i$ is Shannon entropy, and $\lambda \geq 0$ controls the diversity–fit tradeoff.

**Sign convention**: entropy is *subtracted* — minimising $\mathcal{L}$ simultaneously reduces prediction error and *maximises* weight entropy. This is a diversity bonus, not a penalty.

## The Probability Simplex as a Riemannian Manifold

The simplex $\Delta_{n-1}$ carries the **Fisher information metric**:

$$g_{ij}(\theta) = \frac{\delta_{ij}}{\theta_i}, \qquad \langle u, v \rangle_\theta = \sum_i \frac{u_i v_i}{\theta_i}$$

The map $\phi(\theta)_i = \sqrt{\theta_i}$ is an isometry from $(\Delta_{n-1}, g_\text{Fisher})$ to the positive orthant of the unit sphere $S^{n-1}_+$. The geodesic distance is:

$$d_\text{FR}(p, q) = 2\arccos\!\left(\sum_i \sqrt{p_i q_i}\right)$$

where $\sum_i \sqrt{p_i q_i}$ is the Bhattacharyya coefficient (geometric overlap).

The geodesic between $p$ and $q$ is a normalised geometric interpolation: $\gamma(t) = \mathrm{norm}(p^{1-t} \odot q^t)$. It stays strictly inside the simplex — linear interpolation can reach the boundary.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from mirror_linear_regression.geometry import fisher_rao_distance, geodesic
from mirror_linear_regression.utils_math import entropy, effective_number_of_bets

p = np.array([0.7, 0.2, 0.1])
q = np.array([0.1, 0.3, 0.6])
ts = np.linspace(0, 1, 80)
path = np.array([geodesic(p, q, t) for t in ts])
lin  = np.array([(1-t)*p + t*q for t in ts])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#2166ac', '#d6604d', '#1a9850']
for i, (c, lbl) in enumerate(zip(colors, [r'$\theta_1$', r'$\theta_2$', r'$\theta_3$'])):
    axes[0].plot(ts, path[:, i], color=c, lw=2, label=lbl)
    axes[0].scatter([0, 1], [p[i], q[i]], color=c, s=60, zorder=5)
axes[0].set_title(f'Fisher-Rao Geodesic (d={fisher_rao_distance(p,q):.3f})', fontweight='bold')
axes[0].set_xlabel('t'); axes[0].set_ylabel('weight'); axes[0].legend(frameon=False)

axes[1].plot(ts, path.min(axis=1), label='geodesic min weight', color='#2166ac', lw=2)
axes[1].plot(ts, lin.min(axis=1),  label='linear min weight',   color='#d6604d', lw=2, ls='--')
axes[1].axhline(0, color='gray', lw=1, ls=':')
axes[1].set_title('Min Component Along Path', fontweight='bold')
axes[1].set_xlabel('t'); axes[1].legend(frameon=False)

axes[2].plot(ts, [entropy(path[i]) for i in range(len(ts))], color='#2166ac', lw=2, label='geodesic')
axes[2].plot(ts, [entropy(lin[i]/lin[i].sum()) for i in range(len(ts))], color='#d6604d', lw=2, ls='--', label='linear')
axes[2].set_title('Entropy Along Path', fontweight='bold')
axes[2].set_xlabel('t'); axes[2].set_ylabel('H(theta)'); axes[2].legend(frameon=False)
plt.tight_layout(); plt.show()
print(f'Geodesic stays positive: {(path > 0).all()}')
print(f'Linear stays positive:   {(lin > 0).all()}')

## Bregman Divergences and the Mirror Descent Framework

A Bregman divergence induced by a strictly convex generator $\phi$ is:

$$D_\phi(p \| q) = \phi(p) - \phi(q) - \langle \nabla\phi(q),\, p - q \rangle$$

The **mirror descent** update solves a proximal subproblem in the $D_\phi$ geometry:

$$\theta_{t+1} = \arg\min_{\theta \in \Delta} \left\{\eta \langle \nabla\mathcal{L}(\theta_t), \theta \rangle + D_\phi(\theta \| \theta_t) \right\}$$

**With the KL generator** $\phi(p) = \sum_i p_i \log p_i$, this has the closed form:

$$\theta_{t+1,i} \propto \theta_{t,i} \cdot \exp(-\eta\,\nabla_i\mathcal{L})$$

The normalisation *is* the simplex projection in KL geometry — it happens automatically.

### Regret Bounds and Minimax Optimality

| Geometry | Regret Bound | Dimension Term |
| --- | --- | --- |
| KL mirror descent | $G\sqrt{2T\log n}$ | $\sqrt{\log n}$ |
| Euclidean (PGD) | $G\sqrt{2nT}$ | $\sqrt{n}$ |
| Minimax lower bound | $\Omega(G\sqrt{T\log n})$ | optimal |

**Theorem (Cesa-Bianchi & Lugosi, 2006):** No deterministic algorithm can achieve $o(G\sqrt{T\log n})$ regret on $\Delta_{n-1}$ in the worst case. KL mirror descent matches this lower bound to within a constant factor of 4 — it is **minimax optimal in rate**. Euclidean PGD exceeds the lower bound by $\sqrt{n/\log n}$, which grows without bound.

In [ ]:
from mirror_linear_regression import (
    kl_regret_bound, euclidean_regret_bound,
    minimax_lower_bound, euclidean_suboptimality_factor
)

dims = [5, 10, 20, 50, 100, 200, 500, 1000]
T, G = 1000, 1.0

kl_b = [kl_regret_bound(T, n, G) for n in dims]
eu_b = [euclidean_regret_bound(T, n, G) for n in dims]
lb   = [minimax_lower_bound(T, n, G) for n in dims]
sub  = [euclidean_suboptimality_factor(n) for n in dims]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.loglog(dims, kl_b, 'o-', color='#2166ac', lw=2, ms=6,
           label=r'KL: $O(\sqrt{T\log n})$')
ax1.loglog(dims, eu_b, 's-', color='#d6604d', lw=2, ms=6,
           label=r'Euclidean: $O(\sqrt{nT})$')
ax1.loglog(dims, lb,   '^--', color='#1a9850', lw=1.5, ms=5,
           label=r'Lower bound: $\Omega(\sqrt{T\log n})$')
ax1.fill_between(dims, kl_b, eu_b, alpha=0.1, color='#2166ac')
ax1.set_xlabel('Dimension $n$'); ax1.set_ylabel('Regret bound')
ax1.set_title('Regret Bounds (log-log)', fontweight='bold')
ax1.legend(frameon=False, fontsize=9)

ax2.semilogx(dims, sub, 'o-', color='#d6604d', lw=2, ms=6)
ax2.axhline(1, color='#2166ac', ls='--', lw=1.5, label='KL optimality (constant 4x)')
ax2.set_xlabel('Dimension $n$')
ax2.set_ylabel(r'$\sqrt{n/\log n}$')
ax2.set_title('Euclidean Suboptimality Factor', fontweight='bold')
ax2.legend(frameon=False)
plt.tight_layout(); plt.show()

header = f"{'n':>6}  {'KL':>8}  {'LB':>8}  {'KL/LB':>6}  {'Eucl subopt':>12}"
print(header); print('-' * len(header))
for n in [10, 50, 100, 500, 1000]:
    kl = kl_regret_bound(T, n, G)
    lo = minimax_lower_bound(T, n, G)
    print(f'{n:>6}  {kl:>8.2f}  {lo:>8.2f}  {kl/lo:>6.2f}  '
          f'{euclidean_suboptimality_factor(n):>12.2f}x')

## Linear Convergence Under Entropy Regularisation

When $\lambda > 0$, the term $-\lambda H(\theta)$ makes $\mathcal{L}$ **$\lambda$-strongly convex** w.r.t. the KL Bregman divergence. Mirror descent then achieves **linear (exponential) convergence**:

$$\mathcal{L}(\theta_T) - \mathcal{L}(\theta^*) \leq e^{-\mu\eta T}\!\left(\mathcal{L}(\theta_1) - \mathcal{L}(\theta^*)\right)$$

with $\mu = \lambda / \max_i \theta_i$. Iterations to $\varepsilon$-accuracy: $O\!\left(\frac{1}{\mu\eta}\log\frac{1}{\varepsilon}\right)$, versus $O(1/\varepsilon^2)$ without regularisation.

**The tradeoff**: higher $\lambda$ accelerates convergence and increases diversity but biases the solution toward the uniform distribution. Tune $\lambda$ by cross-validation.

In [ ]:
from mirror_linear_regression import MirrorLinearRegression

rng = np.random.RandomState(42)
m, n_feat = 300, 15
X = rng.randn(m, n_feat)
w_true = rng.dirichlet(np.ones(n_feat))
y = X @ w_true + 0.01 * rng.randn(m)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
lam_cfgs = [(0.0,'#d6604d'), (0.005,'#f4a582'), (0.05,'#2166ac'), (0.20,'#053061')]
for lam, color in lam_cfgs:
    model = MirrorLinearRegression(lam=lam, learning_rate=0.1, n_iters=500, tol=0)
    model.fit(X, y)
    axes[0].semilogy(model.loss_history, color=color, lw=1.8, label=f'lam={lam}')
    axes[1].plot(model.entropy_history, color=color, lw=1.8)
    axes[2].bar(range(n_feat), model.weights, color=color, alpha=0.55,
               label=f'lam={lam} H={entropy(model.weights):.2f}')

axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Loss (log scale)')
axes[0].set_title('Convergence Speed vs lambda', fontweight='bold')
axes[0].legend(frameon=False, fontsize=9)
axes[1].axhline(np.log(n_feat), color='gray', ls='--', lw=1, label=f'max H=ln({n_feat})')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('H(theta_t)')
axes[1].set_title('Entropy During Training', fontweight='bold')
axes[1].legend(frameon=False, fontsize=9)
axes[2].plot(range(n_feat), w_true, 'k--', lw=1.5, alpha=0.7, label='true weights')
axes[2].set_title('Final Weight Profiles', fontweight='bold')
axes[2].set_xlabel('feature'); axes[2].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## Summary

1. **OLS has three structural failures** in finance: unconstrained weights, over-concentration under correlated features, no guarantee after post-hoc projection.

2. **The probability simplex is a Riemannian manifold** (Fisher-Rao metric). The KL Bregman geometry achieves the minimax-optimal regret $O(\sqrt{T\log n})$, a factor $\sqrt{n/\log n}$ better than Euclidean methods.

3. **Entropy regularisation** adds strong convexity, converting $O(1/\sqrt{T})$ sublinear convergence into linear convergence. $\lambda$ controls the diversity–fit tradeoff.

**Next → Notebook 2**: Core model implementation and all four optimisers in detail.